# KFP チャレンジラボ 1

このチャレンジラボでは、[kfp_pipeline_prebuilt.ipynb](../solutions/kfp_pipeline_prebuilt.ipynb) で開発したパイプラインを拡張して、Agent Platform Experiments と統合します。

**学習目標:**
1. KFP Artifact の取り扱い方を学ぶ
1. Python 軽量コンポーネントで Artifact 出力を定義する方法を学ぶ
1. Agent Platform Experiments の使用方法を学ぶ

## セットアップ

In [ ]:
from datetime import datetime

from google.cloud import aiplatform

In [ ]:
REGION = "us-central1"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]

### トレーナーイメージのビルド

In [ ]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image_covertype"
IMAGE_TAG = "latest"
TRAINING_CONTAINER_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"
TRAINING_CONTAINER_IMAGE_URI

このノートブックでは、トレーニングコンテナが `us-docker.pkg.dev/{PROJECT_ID}/asl-artifact-repo/trainer_image_covertype` の下にすでに作成されていることを前提としています。以下のコマンドで存在するかどうかを確認できます。

In [ ]:
!gcloud artifacts docker images list \
--include-tags us-docker.pkg.dev/$PROJECT_ID/$ARTIFACT_REGISTRY_DIR/$IMAGE_NAME \
--filter TAGS:$IMAGE_TAG

イメージが存在しない場合は、以下のコメントアウトを解除してビルドしてください。

In [ ]:
# !gcloud builds submit --timeout 15m --tag $TRAINING_CONTAINER_IMAGE_URI trainer_image

モデルのサービング時にトレーニング時に使用した ML フレームワークのバージョンと一致させるために、以下のサービングコンテナをパイプラインに提供する必要があります:

In [ ]:
SERVING_CONTAINER_IMAGE_URI = (
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
)

**注:** トレーニング ML フレームワークのバージョンを変更する場合は、一致するバージョンのサービングコンテナを提供する必要があります（[予測用の事前構築済みコンテナ](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers) を参照）。

## チャレンジラボ 1

このラボでは、[pipelines_prebuild.ipynb](../solutions/kfp_pipeline_prebuilt.ipynb) で開発したパイプラインを拡張して、Agent Platform Experiments と統合します。<br>
Agent Platform Experiments への手動レポート、具体的にはハイパーパラメータ調整の結果（alpha、max_iter の最適値、および最良の accuracy）のロギングに焦点を当てます。これには KFP Metrics オブジェクトを使用します。

この機能を追加するために、`retrieve_best_hptune_component.py` コンポーネントを変更し、出力として Metrics Artifact を追加しましょう。

[pipeline/retrieve_best_hptune_component.py](pipeline/retrieve_best_hptune_component.py) を開き、以下の要素を更新してください:
- **必要なオブジェクトのインポート**: 必要なモジュールの import 文を追加します。
- **Metrics 出力の追加**: コンポーネントの引数内に Metrics 出力アーティファクトを追加します。
- **メトリクスのレポート**: Metrics アーティファクトを使用して、ハイパーパラメータと accuracy を Agent Platform Experiments にレポートするコードを追加します。

**ヒント: `TODO 1` を検索して、更新が必要なセクションを見つけてください。**

また、後述の「パイプラインパッケージのデプロイと実行」セクションで以下のコマンドが更新されている点に注意してください:
- `aiplatform.init()` への実験（experiment）名の追加
- `pipeline.run()` の代わりに実験を指定した `pipeline.submit()` による非同期パイプライン実行

### 参考資料:
- KFP でのアーティファクト出力の追加: https://www.kubeflow.org/docs/components/pipelines/v2/data-types/artifacts/#traditional-artifact-syntax 
- KFP Metrics Artifact: https://kubeflow-pipelines.readthedocs.io/en/latest/source/dsl.html#kfp.dsl.Metrics
- Gemini Enterprise Agent Platform Experiments へのパイプライン実行の追加: https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/experiments/add-pipelinerun-experiment#associate-pipeline-run-with-an-experiment
- Experiments メトリクスレポートの例: https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/experiments/comparing_pipeline_runs.ipynb


### 期待される結果
KFP パイプライン DAG（強調表示されたセクションを拡張）:

<img src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/ee441830-d3e3-45b5-81c6-aea555aa0a7d" width="500"/>

---

Agent Platform Experiments:

<img src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/a55d2792-aac0-4bcb-aca3-fb5f06388417" width="1000"/>

---

<img src="https://github.com/GoogleCloudPlatform/asl-ml-immersion/assets/6895245/431442dc-e72a-4fb1-932c-ab87b5bb96c3" width="1000"/>

## パイプラインのコンパイルと実行

まず、パイプラインコンパイラに渡される環境変数を定義しましょう:

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATA_ROOT = f"{ARTIFACT_STORE}/data"

TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")
BASE_OUTPUT_DIR = f"{ARTIFACT_STORE}/models/{TIMESTAMP}"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT_ID={PROJECT_ID}
%env REGION={REGION}
%env SERVING_CONTAINER_IMAGE_URI={SERVING_CONTAINER_IMAGE_URI}
%env TRAINING_CONTAINER_IMAGE_URI={TRAINING_CONTAINER_IMAGE_URI}
%env TRAINING_FILE_PATH={TRAINING_FILE_PATH}
%env VALIDATION_FILE_PATH={VALIDATION_FILE_PATH}
%env BASE_OUTPUT_DIR={BASE_OUTPUT_DIR}

`ARTIFACT_STORE` が作成されていることを確認し、作成されていない場合は作成しましょう:

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

**注:** アーティファクトストアが事前に作成および適切に設定されていない場合、Agent Platform からのアクセスを許可するために **Cloud Shell** で次のコマンドを実行する必要がある場合があります:

```
PROJECT_ID=$(gcloud config get-value project)
PROJECT_NUMBER=$(gcloud projects list --filter="name=$PROJECT_ID" --format="value(PROJECT_NUMBER)")
gcloud projects add-iam-policy-binding $PROJECT_ID     --member="serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com"     --role="roles/storage.objectAdmin"
```

#### CLI コンパイラを使用してパイプラインをコンパイルする

次のコマンドを使用して、生成した Python ファイルからパイプラインを YAML 記述にコンパイルします:

In [ ]:
PIPELINE_YAML = "covertype_kfp_pipeline.yaml"

In [ ]:
!kfp dsl compile --py pipeline/pipeline.py --output $PIPELINE_YAML

**注:** Python SDK を使用してパイプラインをコンパイルすることもできます:

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

結果としてパイプラインファイルが生成されます。

In [ ]:
!head {PIPELINE_YAML}

### パイプラインパッケージのデプロイと実行

In [ ]:
EXPERIMENT_NAME = "kfp-covertype-experiment"

In [ ]:
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,
    experiment=EXPERIMENT_NAME,
    experiment_tensorboard=False,
)

pipeline = aiplatform.PipelineJob(
    display_name="covertype_kfp_pipeline_challenge_lab",
    template_path=PIPELINE_YAML,
    enable_caching=True,
)

pipeline.submit(experiment=EXPERIMENT_NAME)

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.